In [1]:
import pandas as pd
import requests
import configparser
import mysql.connector

In [2]:
# =========================
# READ CONFIG
# =========================
config = configparser.ConfigParser()
config.read("config.ini")

db_config = {
    "host": config["database"]["host"],
    "user": config["database"]["user"],
    "password": config["database"]["password"],
    "database": config["database"]["database"]
}

In [3]:
# =========================
# MYSQL CONNECTION
# =========================
conn = mysql.connector.connect(**db_config)
cursor = conn.cursor()

# =========================
# API CONFIG
# =========================
url = "https://sikumbang.tapera.go.id/ajax/lokasi/search"

headers = {
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9,id-ID;q=0.8,id;q=0.7",
    "Cache-Control": "no-cache",
    "Connection": "keep-alive",
    "DNT": "1",
    "Pragma": "no-cache",
    "Referer": "https://sikumbang.tapera.go.id/",
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-origin",
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/147.0.0.0 Safari/537.36"
    ),
    "sec-ch-ua": '"Google Chrome";v="147", "Not.A/Brand";v="8", "Chromium";v="147"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
}

In [4]:
# =========================
# CREATE DATAFRAME FUNCTION
# =========================
def create_dataframe(data):

    list_rumah = []

    for dt in data['data']:

        info_rumah = dt['tipeRumah']

        for rumah in info_rumah:

            # =========================
            # INFORMASI RUMAH
            # =========================
            id_rumah = rumah['id']
            status_rumah = rumah['status']
            nama_rumah = rumah['nama']
            harga_rumah = rumah['harga']
            jml_kamar_tidur = rumah['kamarTidur']
            jml_kamar_mandi = rumah['kamarMandi']
            jml_lantai = rumah['jumlahLantai']
            luas_tanah = rumah['luasTanah']
            luas_bangunan = rumah['luasBangunan']

            spek_atap = rumah['spesifikasiAtap']
            spek_dinding = rumah['spesifikasiDinding']
            spek_lantai = rumah['spesifikasiLantai']
            spek_pondasi = rumah['spesifikasiPondasi']

            foto_tampak = None
            foto_denah = None

            if rumah.get('fotoTampak'):
                foto_tampak = 'https://sikumbang.tapera.go.id' + rumah['fotoTampak']

            if rumah.get('fotoDenah'):
                foto_denah = 'https://sikumbang.tapera.go.id' + rumah['fotoDenah']

            # =========================
            # INFORMASI PERUMAHAN
            # =========================
            id_perumahan = dt['idLokasi']
            nama_perumahan = dt['namaPerumahan']
            jenis_perumahan = dt['jenisPerumahan']

            jumlah_unit = dt['jumlahUnit']
            jumlah_unit_komersil = dt['jumlahUnitKomersil']

            longitude = None
            latitude = None

            if dt.get('koordinatPerumahan'):

                koordinat = dt['koordinatPerumahan'].split(',')

                latitude = koordinat[0]
                longitude = koordinat[1]

            # =========================
            # INFORMASI WILAYAH
            # =========================
            wilayah = dt['wilayah']

            kode_wilayah = wilayah['kodeWilayah']
            nama_wilayah = wilayah['namaWilayah']
            nama_provinsi = wilayah['provinsi']
            nama_kabupaten = wilayah['kabupaten']
            nama_kecamatan = wilayah['kecamatan']
            nama_kelurahan = wilayah['kelurahan']

            # =========================
            # UNIQUE ID
            # =========================
            id_unique = str(id_perumahan) + '-' + str(id_rumah)

            informasi_item = [
                id_unique,
                id_rumah,
                status_rumah,
                nama_rumah,
                harga_rumah,
                jml_kamar_tidur,
                jml_kamar_mandi,
                jml_lantai,
                luas_tanah,
                luas_bangunan,
                spek_atap,
                spek_dinding,
                spek_lantai,
                spek_pondasi,
                foto_tampak,
                foto_denah,
                id_perumahan,
                nama_perumahan,
                jenis_perumahan,
                jumlah_unit,
                jumlah_unit_komersil,
                longitude,
                latitude,
                kode_wilayah,
                nama_wilayah,
                nama_provinsi,
                nama_kabupaten,
                nama_kecamatan,
                nama_kelurahan
            ]

            list_rumah.append(informasi_item)

    # =========================
    # DATAFRAME
    # =========================
    columns = [
        'id',

        'id_rumah',
        'status_rumah',
        'nama_rumah',
        'harga_rumah',
        'jml_kamar_tidur',
        'jml_kamar_mandi',
        'jml_lantai',
        'luas_tanah',
        'luas_bangunan',
        'spek_atap',
        'spek_dinding',
        'spek_lantai',
        'spek_pondasi',
        'foto_tampak',
        'foto_denah',

        'id_perumahan',
        'nama_perumahan',
        'jenis_perumahan',
        'jumlah_unit',
        'jumlah_unit_komersil',
        'longitude',
        'latitude',

        'kode_wilayah',
        'nama_wilayah',
        'nama_provinsi',
        'nama_kabupaten',
        'nama_kecamatan',
        'nama_kelurahan'
    ]

    df_rumah = pd.DataFrame(list_rumah, columns=columns)

    return df_rumah

In [5]:
# =========================
# INSERT DATABASE FUNCTION
# =========================
def insert_to_mysql(df):

    sql = """
    INSERT INTO rumah (
        id,
        id_rumah,
        status_rumah,
        nama_rumah,
        harga_rumah,
        jml_kamar_tidur,
        jml_kamar_mandi,
        jml_lantai,
        luas_tanah,
        luas_bangunan,
        spek_atap,
        spek_dinding,
        spek_lantai,
        spek_pondasi,
        foto_tampak,
        foto_denah,
        id_perumahan,
        nama_perumahan,
        jenis_perumahan,
        jumlah_unit_subsidi,
        jumlah_unit_komersil,
        longitude,
        latitude,
        kode_wilayah,
        nama_wilayah,
        nama_provinsi,
        nama_kabupaten,
        nama_kecamatan,
        nama_kelurahan
    )
    VALUES (
        %s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s,
        %s, %s, %s, %s
    )
    ON DUPLICATE KEY UPDATE
        status_rumah = VALUES(status_rumah),
        harga_rumah = VALUES(harga_rumah),
        updated_at = CURRENT_TIMESTAMP
    """

    values = [tuple(x) for x in df.to_numpy()]

    cursor.executemany(sql, values)

    conn.commit()

    print(f"{cursor.rowcount} rows inserted/updated")

In [ ]:
# =========================
# MAIN LOOP
# =========================
page = 1

while True:

    print(f"Scraping page {page}")

    params = {
        "sort": "terbaru",
        "page": page
    }

    response = requests.get(
        url,
        params=params,
        headers=headers
    )

    if response.status_code != 200:
        print("Request failed")
        break

    try:
        data = response.json()

        if len(data['data']) == 0:
            print("No more data")
            break

        # CREATE DATAFRAME
        df_rumah = create_dataframe(data)

        # INSERT MYSQL
        insert_to_mysql(df_rumah)

        page += 1

    except Exception as e:
        print("ERROR:", e)
        break


# =========================
# CLOSE CONNECTION
# =========================
cursor.close()
conn.close()

print("Finished")

Scraping page 1
14 rows inserted/updated
Scraping page 2
30 rows inserted/updated
Scraping page 3
34 rows inserted/updated
Scraping page 4
48 rows inserted/updated
Scraping page 5
54 rows inserted/updated
Scraping page 6
38 rows inserted/updated
Scraping page 7
28 rows inserted/updated
Scraping page 8
42 rows inserted/updated
Scraping page 9
58 rows inserted/updated
Scraping page 10
64 rows inserted/updated
Scraping page 11
38 rows inserted/updated
Scraping page 12
48 rows inserted/updated
Scraping page 13
68 rows inserted/updated
Scraping page 14
64 rows inserted/updated
Scraping page 15
52 rows inserted/updated
Scraping page 16
28 rows inserted/updated
Scraping page 17
38 rows inserted/updated
Scraping page 18
52 rows inserted/updated
Scraping page 19
40 rows inserted/updated
Scraping page 20
86 rows inserted/updated
Scraping page 21
54 rows inserted/updated
Scraping page 22
35 rows inserted/updated
Scraping page 23
20 rows inserted/updated
Scraping page 24
23 rows inserted/updated
S